# Moosic — 07. DBSCAN on the Mega-Cluster Subset

**Question (per instructor suggestion):** the canonical DBSCAN run (`eps=0.20, min_samples=7`) found one dominant mega-cluster (~4286 songs, ~82% of the dataset) plus a small classical/instrumental cluster and noise. That global `eps` had to compromise across the *entire* feature space at once. Does the mega-cluster have its own internal density structure that a single global threshold was too coarse to see?

**Important distinction from an earlier, correctly-rejected idea:** this is NOT re-clustering the mega-cluster with K-Means (that would just reproduce the existing recursive pipeline — see report Section 24, "Considered and Skipped"). This is re-running **DBSCAN itself**, with a fresh `eps` fitted to the subset's own local density — conceptually similar to what HDBSCAN does automatically, done here manually with a technique we already know.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn import set_config
import os

set_config(transform_output="pandas")
os.makedirs("../outputs", exist_ok=True)
RANDOM_STATE = 42

## 2. Load Data & Scale

In [ ]:
df = pd.read_csv("../data/5000_songs.csv")
df.columns = df.columns.str.strip()

features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']

scaler = MinMaxScaler().set_output(transform="pandas")
scaled_all = scaler.fit_transform(df[features])

print(f"Shape: {df.shape}")

## 3. Regenerate the Canonical DBSCAN Run (to Isolate the Mega-Cluster)

In [ ]:
dbscan = DBSCAN(eps=0.20, min_samples=7)
df['dbscan_cluster'] = dbscan.fit_predict(scaled_all)

print(df['dbscan_cluster'].value_counts().sort_index())

## 4. Extract the Mega-Cluster Subset

In [ ]:
mega_cluster_id = df['dbscan_cluster'].value_counts().idxmax()
mega_indices = df[df['dbscan_cluster'] == mega_cluster_id].index
mega_scaled = scaled_all.loc[mega_indices]

print(f"Mega-cluster {mega_cluster_id}: {len(mega_indices)} songs "
      f"({100*len(mega_indices)/len(df):.1f}% of dataset)")

## 5. Fresh k-distance Graph — Local Density, Not Global

Computed on the subset only — this is the whole point of the experiment. The global k-distance graph (notebook `04`) reflects density across the *entire* dataset, including the classical cluster and noise, which pulled the useful `eps` range in a direction that made sense for the whole dataset, not necessarily for this subset alone.


In [ ]:
N_NEIGHBORS = 7

neighbours = NearestNeighbors(n_neighbors=N_NEIGHBORS)
neighbours.fit(mega_scaled)
distances, _ = neighbours.kneighbors(mega_scaled)
kth_distances = distances[:, N_NEIGHBORS - 1]
sorted_distances = sorted(kth_distances)

(
    sns.relplot(kind="line", x=range(len(sorted_distances)), y=sorted_distances, aspect=1.6)
    .set_axis_labels("Points sorted by distance", f"{N_NEIGHBORS}-th Nearest Neighbour Distance")
    .set(title="k-distance Graph — Mega-Cluster Subset Only")
)
plt.savefig("../outputs/07_kdistance_subset.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Parameter Sweep on the Subset

Same sweep technique as notebook `04`, applied to the subset. Range starts narrower than the global sweep, since the subset's own k-distance graph (Step 5) should show a much smaller useful range than the full dataset did.


In [ ]:
def eps_sweep(data, eps_values, min_samples):
    results = []
    for eps in eps_values:
        db = DBSCAN(eps=eps, min_samples=min_samples)
        labels = db.fit_predict(data)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        coverage_pct = 100 * (1 - n_noise / len(labels))

        non_noise = labels != -1
        if n_clusters > 1 and non_noise.sum() > 1:
            sil = silhouette_score(data[non_noise], labels[non_noise])
        else:
            sil = None

        results.append({
            "eps": eps, "clusters": n_clusters, "noise": n_noise,
            "coverage_%": round(coverage_pct, 1),
            "silhouette": round(sil, 3) if sil is not None else None
        })
    return pd.DataFrame(results)

# adjust this range once Step 5's graph shows where the knee actually sits —
# these are a reasonable starting guess, scaled down from the global sweep's range
subset_eps_values = [0.03, 0.05, 0.07, 0.09, 0.11, 0.13, 0.15, 0.18, 0.22, 0.27]

subset_sweep = eps_sweep(mega_scaled, subset_eps_values, min_samples=7)
print(subset_sweep)

## 7. Verdict

**Two honest possible outcomes — both are genuine, reportable findings:**

- **Real sub-structure found** (multiple reasonably-sized clusters, positive silhouette, moderate-to-high coverage, at some eps in the sweep): the mega-cluster was hiding real internal density structure that the global `eps` was too coarse to see. Worth naming what that structure looks like — pull feature averages for each sub-cluster, same technique as the earlier classical/rap pocket investigations.
- **Same pattern repeats** (fragmentation → one dominant sub-blob → collapse, no usable middle): confirms, a further independent way, that this mega-cluster genuinely has no natural internal seams — strengthens the "no viable middle ground" conclusion rather than contradicting it.

*(Check `subset_sweep` above and fill in which outcome actually happened before using this in the presentation.)*


In [ ]:
# Run only if Step 7 shows real sub-structure — pick the best-looking eps from the sweep
# and inspect what the sub-clusters actually are
BEST_SUBSET_EPS = 0.15  # placeholder — set from subset_sweep's actual best result

db_final = DBSCAN(eps=BEST_SUBSET_EPS, min_samples=7)
sub_labels = db_final.fit_predict(mega_scaled)

mega_df = df.loc[mega_indices].copy()
mega_df['sub_cluster'] = sub_labels

print(mega_df['sub_cluster'].value_counts().sort_index())

for cid in sorted(mega_df['sub_cluster'].unique()):
    if cid == -1:
        continue
    group = mega_df[mega_df['sub_cluster'] == cid]
    if len(group) < 20:  # skip tiny fragments for this quick read
        continue
    print(f"\n--- Sub-cluster {cid} ({len(group)} songs) ---")
    print(group[features].mean().round(3))

---
**This closes out the DBSCAN thread.** Fold whichever outcome this produced into the report and presentation — a strengthened "no viable middle ground" finding, or a genuine new sub-structure discovery, either way with real numbers behind it.
